<img src="images/network_prof.png" width="150" alt="Prof looking at a electronic brain" style="float: left; margin-right: 15px; margin-bottom: 10px;"> In this section, we will create real agents to answer students' questions about the network course content.

With the philosophy professor, we saw the basic queries of an LLM, with low-level calls.
This required quite a bit of code, and especially the queries were sequential. When we asked several LLMs to work on the assignment, we had to wait for the response
from the first one to launch the second, when they could have done this work at the same time.

Here is the list of modules we will need in this section

In [2]:
from dotenv import load_dotenv
import os
from pypdf import PdfReader
from IPython.display import Markdown, display 


from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput

import gradio
import asyncio
import requests 


## Retrieve the content of the book "Programmer l'Internet des Objets"

First, we will convert the first pages of the book to text.

In [3]:
book = PdfReader("./PLIDO_BOOK_en.pdf")
book_content = ""
for page in book.pages:
    text = page.extract_text()
    book_content += text

## Give the information to the LLM

<img src="images/agent.png" width="150" alt="One agent" style="float: left; margin-right: 15px; margin-bottom: 10px;">Once the API keys are loaded for several servers (we will use the University of Rennes one by default, but using Gemini is also possible). We provide the URI and Token for the service, then in a second step, we specify the LLM model used. If we used OpenAI by default, these lines would be unnecessary. When calling ```Agent```, in the case of OpenAI, the model name is directly indicated in a string.

When creating the Agent, we give it an identifier for traces, then the instructions that will indicate its role, the limits of responses and the actions it will have to take in certain cases. These instructions also contain the entirety of the book in ASCII. Note that we ask the LLM not to try to answer in detail if the answer is not found in the book.

In [5]:
load_dotenv(override=True)

rennes_api_key = os.getenv("RENNES_API_KEY")
if not rennes_api_key:
    print("RENNES_API_KEY is missing")
    exit(1)

google_api_key = os.getenv("GOOGLE_API_KEY")
if not rennes_api_key:
    print("GOOGLE_API_KEY is missing")
    exit(1)
    
RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/ch@t/api"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=rennes_api_key)
rennes_model  = OpenAIChatCompletionsModel(model="mistralai/Mistral-Small-3.1-24B-Instruct-2503", openai_client=rennes_client)

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model  = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)

ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="dont matter")
ollama_model  = OpenAIChatCompletionsModel(model="mistral:latest", openai_client=ollama_client)


instructions = f"""Here is the content of a book on the Internet of Things

{book_content}

Your responsibility is to represent the author (Laurent Toutain) for interactions with students.
The answers must be professional, clear, and should make students want to engage
in the course, or even choose this training.
If you don't know the answer to questions, respond No."""

book_agent = Agent(name="PLIDO Book Agent", instructions=instructions, model=gemini_model)

An agent is launched using the Python coroutine ```run``` from the ```Runner``` module imported from OpenAI's ```agents``` module. The use of the ```await``` keyword is essential. Here, the difference with a function is minimal, since we only launch one coroutine and wait for its completion before moving to the next instruction.

You can rerun the following cell multiple times by changing the question.

In [ ]:
result = await Runner.run(book_agent, "Is this book a good book to read at the beach?")
display(Markdown(result.final_output))

En tant qu'auteur, je suis peut-être biaisé, mais voici quelques éléments à considérer pour décider si ce livre est un bon choix pour la plage :

**Avantages potentiels pour une lecture à la plage :**

*   **Format modulaire :** Le livre est basé sur un MOOC, ce qui signifie qu'il est divisé en sections relativement courtes. Cela peut être pratique pour lire par petites touches entre deux baignades.
*   **Sujet pertinent :** L'IoT est un domaine en pleine expansion, et comprendre les bases peut être enrichissant.
*   **Niveau technique :** Le livre aborde les aspects techniques de l'IoT, ce qui peut offrir une stimulation intellectuelle.

**Inconvénients potentiels pour une lecture à la plage :**

*   **Technique :** Le sujet peut être un peu lourd pour une lecture de détente à la plage. Il faut être prêt à se concentrer.
*   **Nécessite un support :** Certains exercices peuvent nécessiter un ordinateur et une connexion internet.

**Conclusion :**

Si vous êtes passionné par l'IoT et que vous cherchez une lecture instructive, ce livre peut être un bon choix pour la plage. Cependant, si vous cherchez une lecture purement divertissante, il existe probablement des options plus légères.

C'est parfait, on a notre réponse sur le livre, mais on est un peu frustré, car on a donné les clés au LLM et on sait pas ce qui s'est passé. Un bon réflexe est de demander de tracer la requête. Commme ça, en allant sur le site web qui héberge le LLM on va pouvoir voir comment il a construit sa réponse, ainsi que les échanges protocolaire que l'on a eu entre notre machine et le serveur de LLM.

In [12]:
openai_model = OpenAIChatCompletionsModel(model="gpt-4.1-mini", openai_client=AsyncOpenAI())

openai_agent = Agent(name="OpenAI Book Agent", instructions=instructions, model=openai_model)

with trace("Conseil de lecture"):
    result = await Runner.run(openai_agent, "Est ce que le livre est un bon livre à lire à la plage?")
display(Markdown(result.final_output))

Bonjour ! Ce livre, *Programming the Internet of Things* par Laurent Toutain et ses collègues, est une ressource très complète et technique sur l’Internet des Objets (IoT). Il couvre en détail les protocoles, architectures, technologies et même des exemples pratiques avec du code.

Pour une lecture à la plage, je dirais que ce livre est plutôt adapté à un moment où vous souhaitez vraiment plonger dans le sujet, avec un minimum de concentration et d'envie d'apprendre. Il est dense, assez technique, et demande une certaine mise en condition pour bien assimiler les concepts. Ce n’est pas forcément un livre “léger” pour une lecture de détente au soleil, mais c’est une excellente opportunité d’apprendre beaucoup si vous avez l’envie.

Donc, si vous aimez apprendre même en vacances et que le sujet vous passionne, ce livre est parfait. Si vous préférez quelque chose de plus léger pour la plage, vous pouvez l’alterner avec des lectures plus détendues.

Qu’en pensez-vous ? Je suis là pour vous accompagner dans cette formation quand vous le souhaitez !

SI vous avez accès à OpenAI, vous avez eu une trace assez simple:

<img src="images/trace1.png">

Qui indique que tout le temps de la requête a été consacré à l'utilisation du LLM. 

On voit aussi sur la droite que chaque requête va également contenir dans les instructions, le contenu du livre.

## Messagerie

On peut lier l'interrogation du LLM avec une interface graphique pour que ça soit plus convivial. Pour cela on va utilise le module `gradio` et on va à la fin du code lancer l'interface de chat. A chaque fois que l'utilisateur entrera un message, cela appelera la fonction `chat_fn`. 

In [13]:
async def chat_async(message, history):
    """Fonction async pour l'agent"""
    try:
        result = await Runner.run(book_agent, message)
        return result.final_output
    except Exception as e:
        return f"Erreur: {e}"

def chat_fn(message, history):
    """Fonction sync pour ChatInterface"""
    return asyncio.run(chat_async(message, history)) 

gradio.ChatInterface(chat_fn, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


On peut voir que cela amène à quelques acrobaties dans Python. Gradio appelle une fonction en callback pour traiter la commande de l'utilisateur, et l'interaction avec l'agent se fait par une coroutine. D'où la fonction `chat_fn` qui ne fait qu'appeler la coroutine, attend la fin de son execution grâce à `asyncio.run` et retourne le résultat. 

# Plusieurs agents 

<img src="images/agents.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;"> Nous allons reprendre une structure classique pour voir comment exploiter le parallisme avec `asyncio`, nous allons demander à deux LLM de cogiter sur la même question et l'on prendra la meilleure des deux. Comme nous allons utiliser le modèle ollama en local, il y a peu de chance qu'elle fasse la réponse la plus futée, mais sait-on jamais.

In [15]:
ollama_agent = Agent(name="PLIDO Book Agent by Ollama", instructions=instructions, model=ollama_model)

instructions= """selectionne la réponse la plus claire parmi les différentes options à ces questions
d'étudiants sur un cours. Il faut prendre celui qui te deonnera le plus envie de suivre les cours.
Ne rajoute pas d'explication aux réponses possibles. Repond juste avec la meilleure réponse"""

best_answer  = Agent(name="Response selection", instructions=instructions, model=gemini_model)

async def chat_async(message, history):
    """Fonction async pour les appels au deux agents puis à la sélection"""
    
    results = await asyncio.gather( # lance les deux agents en parallèle
        Runner.run(book_agent, message),
        Runner.run(ollama_agent, message),
    )
    outputs = [result.final_output for result in results]

    answers = "Réponses à la question:\n\n" + "\nRéponse:\n".   join(outputs)
    best = await Runner.run(best_answer, answers)

    return best.final_output

def chat_fn(message, history):
    """Fonction sync pour ChatInterface"""
    return asyncio.run(chat_async(message, history)) 

gradio.ChatInterface(chat_fn, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Amnesie

<img src="images/amnesia.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;"> On peut remarquer que notre chat ne se rappelle de rien. Si vous lui donnez votre nom, il va vous saluer, mais si à la question suivante vous lui demandez quel est votre nom, il ne s'en rappelera pas.

Vous pouvez résoudre partiellement ce problème en injectant dans le prompt utilisateur l'historique de conversation que gradio mémorise dans la variable `history`, mais nous verrons par la suite des techniques beaucoup plus efficaces.

# Envoi de message
<img src="images/telephone.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;">
Nous pouvons demander au LLM d'interagir avec le monde extérieur. pour cela nous allons utiliser une application qui envoie des messages sur votre téléphone portable.

* téléchargez depuis votre magasin d'application (Android ou Apple) l'application `pushover`
* Créez votre compte
* Loguez vous également depuis votre ordinateur
* Recupérez votre clé d'utilisateur qui apparait en haut à droite et stockez là dans la la variable `PUSHOVER_USER_ID` dans le fichier `.env`
* En bas de la page, choisissez *Your Applications* et cliquez sur *Create an Application/Token*
  * Donnez un nom comme *PLIDOagent*, et une fois validé, un token va apparaître
  * Mettez ce token dans la variable `PUSHOVER_TOKEN` dans le fichier `.env`

Le petit programme ci dessus vous permet de teste si ca marche.

In [16]:
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_user_id=os.getenv("PUSHOVER_USER_ID")
pushover_uri ="https://api.pushover.net/1/messages.json"

if not pushover_token or not pushover_user_id:
    print ("User_id or token missing")
    exit(1)

def push(message):
    payload = {"user": pushover_user_id, "token": pushover_token, "message": message}
    x = requests.post(pushover_uri, data=payload)

push("Hello my dear")


Nous allons décrire une fonction qui fait l'interface entre le push et le LLM. Pour cela nous allons utiliser le décorateur de fonctions `@function_tool` definit par openAI.

In [17]:
@function_tool
def send_message(object:str):
    """This function is used to send a message to the book author, to inform that you want to follow his class.
    If a student gives his name and wants to register uses this function to inform me.

    Args:
        - object: Protocol mane.`
    """

    message = f"l'etudiant {object} s'interesse au cours IoT."
    push(message)
    return {"status": "success"}

On peut donc modifier les instructions à notre agent et ajouter lors de sa création l'argument `tools`qui va contenir la liste des programmes qu'il peut appeler. Il va utiliser la description de la *doc string* au debut de la fonction pour comprendre comment l'utiliser.

In [20]:
prospect = False

instructions ="""
Tu veux recruter un étudiant dans le cours IoT. Demande à l'étudiant de fournir ses coordonnées, soit son nom et prénom, soit son adresse de 
courrier électronique. Si tu as l'un des deux, envoie un message à l'auteur du cours grâce à la function send_message.
"""

recruitment_agent  = Agent(name="Recrutement des étudiants", instructions=instructions, model=gemini_model,
                     tools=[send_message])


async def chat_async(message, history):
    """Fonction async pour l'agent"""
    try:
        with trace("recrutement"):
            result = await Runner.run(recruitment_agent, message)
        return result.final_output
    except Exception as e:
        return f"Erreur: {e}"

def chat_fn(message, history):
    """Fonction sync pour ChatInterface"""
    return asyncio.run(chat_async(message, history)) 

gradio.ChatInterface(chat_fn, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


En utilisant un agent sur OpenAI, on peut obtenir une trace de l'échange,

<img src="images/trace2.png">

ou l'étudiant fournit son nom:
* dans input on voir les instructions et le message de l'utilisateur
* dans outpout on voit l'appel explicite à un outil `send_message` avec les arguments. L'outil s'execute bien sur votre machine, mais le LLM distant le pilote.

Dans un second temps, vous retournez l'agent, le résultat de l'appel à la fonction `send_message`.

Puis dans un troisième temps, un message est généré pour informer l'utilisateur que ca s'est bien passé.

# De la vraie Agentique AI

Bon, là on peut interfacer des fonctions avec un LLM, donc l'étape d'après est de faire de même avec les Agents. Dans le code précédent, le cheminement est alogirthmique et guidé par le code Python que l'on a écrit:
* Deux Agents sont appelés pour fournir des réponses
* un troisième Agent analyse les réponses, choisi la meilleure et s'il détecte une référence à un protocole, envoie une alerte au professeur.

On va donc changer la logique du code, en transformant les deux Agents responsable des réponses en function et le troisième Agent va orchestrer l'ensemble du processus, c'est a dire appeler les fonctions (ex Agent) pour les réponse et s'il detecte un protocole, envoyer un message.

In [ ]:
tool_answer1 = book_agent.as_tool(tool_name="Book_agent", tool_description="answer to user questions")
tool_answer2 = recruitment_agent.as_tool(tool_name="Recruitment_agent", tool_description="Ask the student's name")

tools = [tool_answer1, tool_answer2]

instructions ="""
Tu gères les inscriptions au cours IoT. Les étudiants vont te poser des questions sur le contenu du cours, 
qui est également donné dans le livre. Le book_agent peut répondre à ces questions techniques. Si 
l'étudiants veut s'inscrire utilise le Recruitment_agent l'envoyer au professeur grâce à la function send_message.
"""

global_agent = Agent("Global Agent", instructions=instructions, tools=tools, model=openai_model)

async def chat_async(message, history):
    """Fonction async pour les appels au deux agents puis à la sélection"""
    
    with trace("agentique"):
        result = await Runner.run(global_agent, message)

    return result.final_output
    
def chat_fn(message, history):
    """Fonction sync pour ChatInterface"""
    return asyncio.run(chat_async(message, history)) 

gradio.ChatInterface(chat_fn, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
